In [7]:
import numpy as np
import pickle
import json

from keras.layers import Flatten, Dense, Dropout
from keras.models import Model

from tensorflow.keras.applications import ResNet50, EfficientNetB0, MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping
from keras.optimizers import Adam

from tensorflow.keras.saving import save_model

In [3]:
# datasets 
with open('../datasets/LID_fg_dataset.pkl', "rb") as f:
    data = pickle.load(f)

(x_train, y_train), (x_validation, y_validation), (x_test,y_test) = data
x_train =x_train.reshape(-2, 128,128,3)
x_validation =x_validation.reshape(-2, 128,128,3)
x_test =x_test.reshape(-2, 128,128,3)

x_train = x_train.astype('float')/255.0
x_validation = x_validation.astype('float')/255.0
x_test = x_test.astype('float')/255.0


In [4]:


def convNet_classification(convNet: str, num_classes: int):
    '''ConvNet (str) - Escolha entre:
                       ResNet50, EfficientNetB0 ou MobileNetV2
       num_classes (int) - Número de classes para a camada de saída'''
    
    # Carregar o modelo base sem a camada fully-connected (include_top=False)
    if convNet == 'ResNet50':
        base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
    elif convNet == 'EfficientNetB0':
        base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
    elif convNet == 'MobileNetV2':
        base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
    else:
        raise ValueError(f"ConvNet '{convNet}' não encontrado.")
    
    # Adicionar camadas fully-connected personalizadas para classificação
    x = base_model.output
    x = Flatten()(x)
    x = Dense(128, activation='relu')(x)  # Camada totalmente conectada com 128 unidades e ReLU
    x = Dropout(0.5)(x)  # Dropout para evitar overfitting
    predictions = Dense(num_classes, activation='softmax')(x)  # Camada de saída

    # Definir o modelo final
    model = Model(inputs=base_model.input, outputs=predictions)

    # Congelar as camadas convolucionais do modelo base
    for layer in base_model.layers:
        layer.trainable = False

    # Compilar o modelo
    model.compile(
        optimizer='Adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model





In [5]:
EffBo_model = convNet_classification('EfficientNetB0', num_classes=10)

In [6]:
history = EffBo_model.fit(x_train, y_train, epochs=10, validation_data=(x_validation, y_validation))


Epoch 1/10
363/363 ━━━━━━━━━━━━━━━━━━━━ 64s 166ms/step - accuracy: 0.0952 - loss: 2.7018 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 2/10
363/363 ━━━━━━━━━━━━━━━━━━━━ 62s 172ms/step - accuracy: 0.0968 - loss: 2.3027 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 3/10
363/363 ━━━━━━━━━━━━━━━━━━━━ 66s 180ms/step - accuracy: 0.1022 - loss: 2.3027 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 4/10
363/363 ━━━━━━━━━━━━━━━━━━━━ 67s 184ms/step - accuracy: 0.0992 - loss: 2.3027 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 5/10
363/363 ━━━━━━━━━━━━━━━━━━━━ 67s 185ms/step - accuracy: 0.0915 - loss: 2.3028 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 6/10
363/363 ━━━━━━━━━━━━━━━━━━━━ 69s 190ms/step - accuracy: 0.0990 - loss: 2.3027 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 7/10
363/363 ━━━━━━━━━━━━━━━━━━━━ 67s 184ms/step - accuracy: 0.0976 - loss: 2.3027 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 8/10
363/363 ━━━━━━━━━━━━━━━━━━━━ 68s 187ms/step - accuracy: 0.0997 - loss: 2

In [10]:
for layer in EffBo_model.layers[-20:]:  # Descongele as últimas 20 camadas, por exemplo
    layer.trainable = True

EffBo_model.compile(
    # optimizer=models.optimizers.Adam(1e-5),  # Use uma taxa de aprendizado menor
    optimizer=Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_fine = EffBo_model.fit(x_train, y_train, epochs=34, validation_data=(x_validation, y_validation))


Epoch 1/34
363/363 ━━━━━━━━━━━━━━━━━━━━ 73s 188ms/step - accuracy: 0.0942 - loss: 2.3029 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 2/34
363/363 ━━━━━━━━━━━━━━━━━━━━ 69s 190ms/step - accuracy: 0.1016 - loss: 2.3027 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 3/34
363/363 ━━━━━━━━━━━━━━━━━━━━ 72s 198ms/step - accuracy: 0.0982 - loss: 2.3028 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 4/34
363/363 ━━━━━━━━━━━━━━━━━━━━ 71s 194ms/step - accuracy: 0.1013 - loss: 2.3026 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 5/34
363/363 ━━━━━━━━━━━━━━━━━━━━ 72s 197ms/step - accuracy: 0.1022 - loss: 2.3028 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 6/34
363/363 ━━━━━━━━━━━━━━━━━━━━ 75s 206ms/step - accuracy: 0.0964 - loss: 2.3027 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 7/34
363/363 ━━━━━━━━━━━━━━━━━━━━ 71s 196ms/step - accuracy: 0.1047 - loss: 2.3026 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 8/34
363/363 ━━━━━━━━━━━━━━━━━━━━ 72s 198ms/step - accuracy: 0.0982 - loss: 2

In [4]:
import pandas as pd
import numpy as np

In [5]:
df= pd.read_csv('../models/metricas.csv', sep=';')
df

,CNN Arquitecture,# Params,Size (MB),Accuracy,Precision,Recall,F1-Score
0,ResNet-50_v1,"36,174,880",138.00,0.100000,0.010204,0.100000,0.018519
1,ResNet-50_v2,"36,174,880",138.00,0.100690,0.012895,0.100690,0.020263
2,ResNet-50_v3,"36,174,880",138.00,0.108966,0.051572,0.108966,0.030490
3,ResNet-50_v4,"36,174,880",138.00,0.113793,0.064894,0.113793,0.038253
4,ResNet-50_v5,"36,174,880",138.00,0.104828,0.264763,0.104828,0.041823
5,MobileNet-V2_v1,"10,126,560",38.63,0.124138,0.229137,0.124138,0.056273
6,MobileNet-V2_v2,"10,126,560",38.63,0.144828,0.152338,0.144828,0.084834
7,MobileNet-V2_v3,"10,126,560",38.63,0.117241,0.266516,0.117241,0.079623
8,MobileNet-V2_v4,"10,126,560",38.63,0.113793,0.321179,0.113793,0.043352
9,MobileNet-V2_v5,"10,126,560",38.63,0.151034,0.324258,0.151034,0.071674


In [45]:
df[15:20]

,CNN Arquitecture,# Params,Size (MB),Accuracy,Precision,Recall,F1-Score
15,f_graphs_v1,"19,440,800",74.16,0.986897,0.986990,0.986897,0.986903
16,f_graphs_v2,"19,440,800",74.16,0.983448,0.983539,0.983448,0.983435
17,f_graphs_v3,"19,440,800",74.16,0.982759,0.982924,0.982759,0.982764
18,f_graphs_v4,"19,440,800",74.16,0.984828,0.984924,0.984828,0.984822
19,f_graphs_v5,"19,440,800",74.16,0.987586,0.983045,0.982759,0.982797


In [55]:
#'CNN Arquitecture', '# Params', 'Size (MB)', 'Accuracy', 'Precision','Recall', 'F1-Score'
vals = df['Accuracy'][15:20].values
vals

array([0.98689655, 0.98344828, 0.98275862, 0.98482759, 0.9875862 ])

In [56]:
round(np.std(vals),3)

0.002